# 10 - Full Assessment Pipeline

End-to-end example: material selection → fatigue assessment → crash check → report generation.

In [ ]:
from weldfatigue.fatigue.assessment import FatigueAssessment
from weldfatigue.shock.crash_assessment import CrashAssessment
from weldfatigue.materials.database import MaterialDatabase
from weldfatigue.reporting.html_report import HTMLReportGenerator
import numpy as np

## Step 1: Select Material

In [ ]:
db = MaterialDatabase()
mat = db.get("DP600")
print(f"Material: {mat.name}")
print(f"  Family: {mat.family}")
print(f"  σy = {mat.yield_strength} MPa")
print(f"  σu = {mat.ultimate_strength} MPa")

## Step 2: Fatigue Assessment (Nominal Method)

In [ ]:
fatigue = FatigueAssessment()

# Single block assessment
result = fatigue.run(
    method="nominal",
    material_name="DP600",
    weld_type="fillet",
    load_type="tension",
    stress_range=80.0,
    num_cycles=2_000_000,
)

print(f"Method: {result['method']}")
print(f"FAT class: {result['fat_class']}")
sr = result['single_block_result']
print(f"Allowable cycles: {sr['allowable_cycles']:.2e}")
print(f"Safety factor: {sr['safety_factor']:.3f}")
print(f"Status: {sr['status']}")

## Step 3: Variable Amplitude Assessment

In [ ]:
spectrum = [
    (120.0, 100_000),
    (100.0, 500_000),
    (80.0, 1_000_000),
    (60.0, 3_000_000),
]

va_result = fatigue.run(
    method="nominal",
    material_name="DP600",
    weld_type="fillet",
    load_type="tension",
    stress_range=120.0,
    num_cycles=100_000,
    load_spectrum=spectrum,
    variable_amplitude=True,
)

miner = va_result['miner_result']
print(f"Miner total damage: {miner['total_damage']:.4f}")
print(f"Status: {miner['status']}")
print(f"Damage per block: {[f'{d:.6f}' for d in miner['damage_per_block']]}")

## Step 4: Crash Assessment

In [ ]:
crash = CrashAssessment()

# Simulate force-displacement from a crush tube
displacement = np.linspace(0, 100, 500)
force = 15000 * (1 - np.exp(-0.05 * displacement)) + 2000 * np.sin(0.1 * displacement)

full_crash = crash.run_full_assessment(
    material_name="DP600",
    strain_rate=500.0,
    weld_check={
        "criterion": "stress_based",
        "sigma_perp": 150,
        "tau_perp": 80,
        "tau_parallel": 50,
        "fu": mat.ultimate_strength,
    },
    force_displacement={
        "force": force,
        "displacement": displacement,
        "mass": 2.5,
    },
)

dyn = full_crash['dynamic_material']
print(f"Dynamic yield: {dyn['dynamic_yield']:.0f} MPa (DIF={dyn['enhancement_factor']:.3f})")

weld = full_crash['weld_failure']
print(f"Weld check: utilization={weld['utilization']:.3f} → {weld['status']}")

energy = full_crash['energy']
print(f"Energy: {energy['total_energy']:.0f} J, CFE={energy['crush_force_efficiency']:.3f}")
print(f"SEA: {energy['specific_energy_absorption']:.0f} J/kg")

## Step 5: Visualize Results

In [ ]:
from weldfatigue.reporting.plots import FatiguePlots, ShockPlots
from weldfatigue.fatigue.sn_curve import SNCurve

sn = SNCurve(fat_class=result['fat_class'], material_type="steel")
fig = FatiguePlots.sn_curve_plotly(sn, highlight_point=(2e6, 80))
fig.show()

In [ ]:
fig2 = ShockPlots.force_displacement_plotly(
    force, displacement,
    metrics={'mean_force': energy['mean_force'], 'peak_force': energy['peak_force']}
)
fig2.show()

## Step 6: Generate HTML Report

In [ ]:
from pathlib import Path

gen = HTMLReportGenerator()

fatigue_html = gen.generate_fatigue_report(
    project_name="EV Battery Enclosure - Weld Joint A3",
    author="C-Power Engineering",
    date="2024-06-15",
    material_info={
        "Name": mat.name,
        "Family": mat.family,
        "Yield Strength": f"{mat.yield_strength} MPa",
        "Ultimate Strength": f"{mat.ultimate_strength} MPa",
    },
    fatigue_results=[{
        "method": sr['method'],
        "fat_class": result['fat_class'],
        "applied_cycles": sr['applied_cycles'],
        "allowable_cycles": sr['allowable_cycles'],
        "damage_ratio": sr['applied_cycles'] / sr['allowable_cycles'],
        "safety_factor": sr['safety_factor'],
        "status": sr['status'],
    }],
    miner_result=miner,
    output_path=Path("fatigue_report.html"),
)
print("Fatigue report saved to fatigue_report.html")

shock_html = gen.generate_shock_report(
    project_name="EV Battery Enclosure - Crash Load Case",
    author="C-Power Engineering",
    date="2024-06-15",
    material_info={"Name": mat.name, "Family": mat.family},
    crash_result=dyn,
    weld_result=weld,
    energy_result=energy,
    output_path=Path("shock_report.html"),
)
print("Shock report saved to shock_report.html")